# 03 · Models and experiments

**You'll learn:** how much each design choice matters: which LLM answers, how the filing was chunked, and how retrieval works. Then you'll measure quality with a small evaluation.

**Time:** about 10 minutes · **Cost:** a few cents · **Runs on its own:** run the two setup cells, then **any section in any order**; each section has its own imports.

```mermaid
flowchart LR
  Q["Question"] --> R{"Retrieval strategy<br/>Section C"}
  R --> COL{"Chunk collection<br/>Section B"}
  COL --> L{"LLM<br/>Section A"}
  L --> E["Answer → scored by<br/>Section D"]
```

| Section | Question it answers | Code |
|---|---|---|
| A | Does the answer change with OpenAI vs Gemini? | `core/llm.py` |
| B | Does layout-aware chunking help on tables? | `sec_filings_basic` vs `sec_filings_advanced` |
| C | Do smarter retrieval strategies find better chunks? | `core/retrieval_strategies.py` |
| D | How good is the pipeline, in numbers? | `eval/` |

---
## Setup

### Setup
The next cell connects the notebook to the project. Read its first lines: `USE_EMBEDDED_QDRANT` picks where the database runs (Docker server, or embedded with no Docker). It prints which LLM, embedding model and database this run uses, all taken from your `.env`.

In [1]:
# ── Settings ────────────────────────────────────────────────────────────
# False: use the Qdrant server from `docker compose up -d`.
# True:  no Docker. Qdrant runs inside this notebook and stores data in data/qdrant_local/.
#        Restart the kernel after changing this, and run one notebook at a time
#        (only one process can open the embedded folder).
USE_EMBEDDED_QDRANT = False

# Setup: run from the repo root so `core`/`ingestion` imports and data/ paths work.
import logging
import os
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.WARNING)
if USE_EMBEDDED_QDRANT:
    os.environ["QDRANT_PATH"] = "data/qdrant_local"
warnings.filterwarnings("ignore", message="Qdrant client version")  # harmless client/server version notice

import pandas as pd

import core  # noqa: F401  (loads .env)
import core.embeddings as emb
import core.llm as llm
from core.clients import COLLECTION_NAME, QDRANT_PATH, QDRANT_URL, get_qdrant_client

pd.set_option("display.max_colwidth", 120)
print(f"LLM:        {llm.PROVIDER} / {llm.MODEL}")
print(f"Embeddings: {emb.PROVIDER} / {emb.MODEL}")
print(f"Qdrant:     {'embedded @ ' + QDRANT_PATH if QDRANT_PATH else 'server @ ' + QDRANT_URL}")

LLM:        gemini / gemini-3.5-flash
Embeddings: gemini / gemini-embedding-001
Qdrant:     server @ http://localhost:6333


In [2]:
# Is the vector database running?
try:
    names = [c.name for c in get_qdrant_client().get_collections().collections]
    print("Qdrant is up. Collections:", names or "(none yet)")
except Exception as exc:
    raise SystemExit(
        "Qdrant is not reachable. Either open Docker Desktop and run `docker compose up -d`, "
        "or set USE_EMBEDDED_QDRANT = True in the first cell and restart the kernel."
    ) from exc

# Make sure the data is ready, so this notebook runs on its own. prepare_data() fetches,
# chunks and indexes only what's missing: a few minutes the very first time, then seconds.
from ingestion.prepare import prepare_data

for name, count in prepare_data().items():
    print(f"{name}: {count} points")

Qdrant is up. Collections: ['sec_filings_advanced', 'sec_filings_advanced_gemini', 'sec_filings_basic', 'sec_filings_basic_gemini']
sec_filings_basic_gemini: 168 points
sec_filings_advanced_gemini: 310 points


---
## Section A · OpenAI vs Gemini for the answer

The provider normally comes from `.env`. Here `use_llm()` switches it in memory, so the same question, retrieval and prompt go to each model. Providers without an API key in `.env` are skipped.

**What to look for:** whether both models cite the same section, and how time and cost compare.

> Only the **LLM** can be swapped like this. Changing the **embedding** model needs a re-index (`REINDEX = True` in notebook 01).

In [3]:
import time

from core.rag import answer_question

QUESTION = "Why did Greater China net sales decrease in fiscal 2024?"


def use_llm(provider, model=None):
    llm.PROVIDER = provider
    llm.MODEL = model or llm._DEFAULT_MODEL[provider]


KEYS = {"openai": "OPENAI_API_KEY", "gemini": "GEMINI_API_KEY"}
original = (llm.PROVIDER, llm.MODEL)

rows = []
for provider in ("openai", "gemini"):
    if not os.environ.get(KEYS[provider]):
        print(f"skipping {provider}: no {KEYS[provider]} in .env")
        continue
    use_llm(provider)
    start = time.time()
    try:
        result = answer_question(QUESTION)
    except Exception as exc:
        print(f"{provider} failed: {str(exc)[:150]}")
        continue
    rows.append({
        "model": f"{provider}/{llm.MODEL}",
        "seconds": round(time.time() - start, 1),
        "citations": len(result.citations),
        "cost_usd≈": round(result.cost_usd, 5),
        "answer": result.answer,
    })

use_llm(*original)
pd.DataFrame(rows)

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

,model,seconds,citations,cost_usd≈,answer
0,openai/gpt-4o,3.4,1,0.00702,"Greater China net sales decreased in fiscal 2024 primarily due to lower net sales of iPhone and iPad. Additionally, ..."
1,gemini/gemini-3.5-flash,4.2,1,0.00111,Greater China net sales decreased during fiscal 2024 compared to 2023 primarily due to lower net sales of iPhone and...


---
## Section B · Basic vs advanced chunks

Same question, same search, same LLM; only the collection changes. Gross margin lives in tables, which the basic chunker flattens and the advanced one keeps.

**What to look for:** which sections come back on top, and whether the answer gives both years.

In [4]:
from core.rag import answer_question
from core.retrieval import hybrid_search
from ingestion.index_comparison_collections import ADVANCED_COLLECTION, BASIC_COLLECTION

TABLE_QUESTION = "What was Apple's gross margin percentage in 2024 vs. 2023?"

for collection in (BASIC_COLLECTION, ADVANCED_COLLECTION):
    result = answer_question(TABLE_QUESTION, collection_name=collection)
    top = hybrid_search(TABLE_QUESTION, top_k=3, collection_name=collection)
    print(f"===== {collection}")
    print("top sections:", [c.section[:45] for c in top])
    print("answer:", result.answer, "\n")

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

===== sec_filings_basic_gemini
top sections: ['Item 7.    Management’s Discussion and Analys', 'Item 7.    Management’s Discussion and Analys', 'Item 8.    Financial Statements and Supplemen']
answer: Apple's total gross margin percentage was 46.2% in 2024 compared to 44.1% in 2023. 

By category:
* **Products gross margin percentage:** 37.2% in 2024 vs. 36.5% in 2023.
* **Services gross margin percentage:** 73.9% in 2024 vs. 70.8% in 2023. 



Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

===== sec_filings_advanced_gemini
top sections: ["Item 7.    Management's Discussion and Analys", "Item 7.    Management's Discussion and Analys", "Item 7.    Management's Discussion and Analys"]
answer: In 2024, Apple's total gross margin percentage was 46.2%, compared to 44.1% in 2023. 

Broken down by category:
* **Products gross margin percentage:** 37.2% in 2024 vs. 36.5% in 2023.
* **Services gross margin percentage:** 73.9% in 2024 vs. 70.8% in 2023. 



---
## Section C · Retrieval strategies

`hybrid_search()` searches with the user's exact words. The strategies in `core/retrieval_strategies.py` try to search better:

| Strategy | Idea | Extra cost |
|---|---|---|
| `baseline` | plain hybrid search | none |
| `query_rewrite` | the LLM rewrites the question into a cleaner search query | 1 LLM call |
| `hyde` | the LLM writes a *hypothetical* 10-K passage that would answer it, and that passage is searched for | 1 LLM call |
| `multi_query` | the LLM writes several rephrasings; their results are fused with RRF | 1 LLM call + extra searches |
| `rerank` | fetch 3 × k candidates, then a cross-encoder model re-scores each (question, chunk) pair | local model (downloaded on first use) |

**What to look for:** whether the top sections change, and the time each strategy adds.

In [5]:
import time

from core.retrieval_strategies import STRATEGIES

QUESTION = "Why did Greater China net sales decrease in fiscal 2024?"

rows = []
for name, strategy in STRATEGIES.items():
    start = time.time()
    try:
        found = strategy(QUESTION, top_k=5)
        sections = [c.section[:35] for c in found[:3]]
    except Exception as exc:
        sections = [f"error: {str(exc)[:80]}"]
    rows.append({"strategy": name, "seconds": round(time.time() - start, 1), "top 3 sections": sections})
pd.DataFrame(rows)

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

,strategy,seconds,top 3 sections
0,baseline,0.7,"[Item 7. Management's Discussion , Item 7. Management's Discussion , Item 8. Financial Statements and]"
1,query_rewrite,7.4,"[Item 7. Management's Discussion , Item 7. Management's Discussion , Item 7. Management's Discussion ]"
2,hyde,8.4,"[Item 1A. Risk Factors > The Comp, Item 8. Financial Statements and, Item 7. Management's Discussion ]"
3,multi_query,14.0,"[Item 7. Management's Discussion , Item 8. Financial Statements and, Item 7. Management's Discussion ]"
4,rerank,1.0,"[Item 7. Management's Discussion , Item 7. Management's Discussion , Item 7. Management's Discussion ]"


---
## Section D · Mini evaluation

Opinions about answers don't scale; numbers do. `eval/golden_dataset.py` has 12 hand-labeled questions with the section that should be found. This scores a few of them:

| Metric | Meaning |
|---|---|
| `hit` | did an expected section appear in the top-k? |
| `mrr` | `1 / rank` of the first relevant chunk (1.0 = ranked first) |
| `recall` | share of the question's expected sections found |
| `correctly_refused` | for "not found" questions: did the app decline? |

`N` keeps it cheap. The full run, with optional LLM-judge scores for faithfulness and correctness, is `python -m eval.run_eval --pipeline advanced`.

In [6]:
from eval.golden_dataset import GOLDEN_SET
from eval.run_eval import run_example, summarize

N = 2
examples = [e for e in GOLDEN_SET if e.should_find][: N - 1] + [e for e in GOLDEN_SET if not e.should_find][:1]
results = [run_example(e, collection=None, strategy="baseline", top_k=8, use_judge=False) for e in examples]

display(pd.DataFrame(results)[["id", "category", "hit", "mrr", "recall", "correctly_refused", "cost_usd"]])
summarize(results)

Embedding with Gemini:   0%|          | 0/1 [00:00<?, ?chunk/s]

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 1h2m32.486812062s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See http

---
## Recap and things to try

- **A:** the LLM mostly changes wording, speed and cost; retrieval decides what it can say.
- **B:** chunking matters most for tables.
- **C:** smarter strategies cost extra calls; check they find better sections before paying for them.
- **D:** measure before and after any change.

**Try this:** in Section D set `strategy="rerank"` and compare `mrr`; in Section B try *How did Services net sales compare to iPhone net sales in 2024?*

**Next:** `04_agentic_rag.ipynb`: let the LLM decide what to search for.